# Medicare Part D ve LEIE Veri Setlerini Birleştirme

Bu notebook, LEIE (List of Excluded Individuals/Entities) veri setini Medicare Part D verileriyle birleştirerek fraud etiketli veri seti oluşturur.

## Veri Kaynakları:
- **LEIE**: Dolandırıcılık nedeniyle hariç tutulan sağlık hizmeti sağlayıcıları
- **Medicare Part D**: Reçete ilaç verileri (2017-2019)

## Adımlar:
1. LEIE veri setini işle
2. Medicare Part D veri setini işle
3. NPI numarasına göre birleştir
4. Fraud etiketleri oluştur
5. Veriyi küçült ve kaydet

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import warnings
warnings.filterwarnings('ignore')

# Output klasörünü oluştur
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../data', exist_ok=True)

## 1. LEIE Veri Setini İşleme

LEIE (List of Excluded Individuals/Entities), dolandırıcılık veya suistimal nedeniyle Medicare/Medicaid programlarından hariç tutulan sağlayıcıların listesidir.

In [2]:
# LEIE veri setini yükle
try:
    leie = pd.read_csv('../data/Leie_Updated.csv')
    print(f"LEIE veri seti yüklendi: {leie.shape}")
except FileNotFoundError:
    print("⚠ LEIE_UPDATED.csv bulunamadı.")
    print("  → https://oig.hhs.gov/exclusions/exclusions_list.asp adresinden indirin.")
    leie = None

LEIE veri seti yüklendi: (82388, 18)


In [3]:
if leie is not None:
    # Boyutları göster
    print(f"Veri seti boyutu: {leie.shape}")
    
    # Sütun isimlerini yazdır
    print("\nSütun İsimleri:")
    print(leie.columns.tolist())
    
    # İlk 5 satır
    leie.head()

Veri seti boyutu: (82388, 18)

Sütun İsimleri:
['LASTNAME', 'FIRSTNAME', 'MIDNAME', 'BUSNAME', 'GENERAL', 'SPECIALTY', 'UPIN', 'NPI', 'DOB', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'EXCLTYPE', 'EXCLDATE', 'REINDATE', 'WAIVERDATE', 'WVRSTATE']


In [4]:
if leie is not None:
    # Paper'dan alınan fraud ile ilişkili hariç tutma kodları
    exclusion_codes = ['1128a1', '1128a2', '1128a3', '1128b4', '1128b7', '1128c3gi', '1128c3gii']

    # EXCLTYPE sütununa göre filtrele
    filtered_leie = leie[leie.EXCLTYPE.isin(exclusion_codes)]
    print(f"Filtrelenmiş LEIE boyutu: {filtered_leie.shape}")
    print(f"\nHariç tutma kodu dağılımı:")
    print(filtered_leie['EXCLTYPE'].value_counts())

Filtrelenmiş LEIE boyutu: (72702, 18)

Hariç tutma kodu dağılımı:
EXCLTYPE
1128b4    32923
1128a1    25299
1128a2     7991
1128a3     5756
1128b7      733
Name: count, dtype: int64


In [5]:
if leie is not None:
    # EXCLDATE sütununu datetime'a çevir (format: YYYYMMDD)
    filtered_leie['EXCLDATE'] = pd.to_datetime(filtered_leie.EXCLDATE, format='%Y%m%d')
    print("EXCLDATE datetime'a çevrildi.")
    print(f"Tarih aralığı: {filtered_leie['EXCLDATE'].min()} - {filtered_leie['EXCLDATE'].max()}")

EXCLDATE datetime'a çevrildi.
Tarih aralığı: 1978-06-02 00:00:00 - 2025-12-18 00:00:00


In [6]:
def calculate_exclusion_end(exclusion_date, end_year): 
    """
    Hariç tutma bitiş yılını hesaplar.
    
    Args:
        exclusion_date: Hariç tutma tarihi
        end_year: Bitiş yılı
    
    Returns:
        Hesaplanmış bitiş yılı
    """
    month = exclusion_date.month  # Ayı çıkar
    if month > 6:  # Temmuz-Aralık arası
        return end_year + 1 
    else:  # Ocak-Haziran arası
        return end_year

## 2. Medicare Part D Veri Setini İşleme

In [7]:
# Medicare Part D veri setini yükle
try:
    partd_main = pd.read_csv('../data/Combined_Medicare_2017_2019.csv')
    print(f"Medicare Part D yüklendi: {partd_main.shape}")
except FileNotFoundError:
    print("⚠ Combined_Medicare_2017_2019.csv bulunamadı.")
    print("  → Önce 'medicare_dataset_filter_and_combining.ipynb' çalıştırın.")
    partd_main = None

Medicare Part D yüklendi: (21932351, 8)


In [8]:
if partd_main is not None:
    print(f"Veri seti boyutu: {partd_main.shape}")
    print("\nSütun İsimleri:")
    print(partd_main.columns.tolist())
    partd_main.head()

Veri seti boyutu: (21932351, 8)

Sütun İsimleri:
['Prscrbr_NPI', 'Prscrbr_Type', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst', 'Tot_Benes', 'Source_Year']


In [9]:
if partd_main is not None:
    # Birleştirme için sütun adlarını düzenle
    partd_main.rename({'Prscrbr_NPI':'npi'}, axis=1, inplace=True)
    
if leie is not None:
    # LEIE'de de aynı şekilde düzenle
    filtered_leie.rename({'NPI':'npi'}, axis=1, inplace=True)
    
print("Sütun adları düzenlendi (npi olarak birleştirilecek).")

Sütun adları düzenlendi (npi olarak birleştirilecek).


In [10]:
if partd_main is not None:
    # Source_Year sütununu DATA_YEAR olarak yeniden adlandır
    partd_main.rename(columns={'Source_Year': 'DATA_YEAR'}, inplace=True)
    
    # Tot_Benes'daki 0'ları 5 ile değiştir (gizlilik nedeniyle)
    partd_main.Tot_Benes = partd_main.Tot_Benes.replace(0, 5)
    print("Tot_Benes'daki 0 değerleri 5 ile değiştirildi.")

Tot_Benes'daki 0 değerleri 5 ile değiştirildi.


In [11]:
if partd_main is not None:
    # NPI, Prscrbr_Type, DATA_YEAR'a göre grupla ve istatistikleri hesapla
    partd_main = partd_main.groupby(['npi','Prscrbr_Type', 'DATA_YEAR']).agg({
        'Tot_Benes': ['sum', 'mean', 'median', np.std, 'min', 'max'],
        'Tot_Clms': ['sum', 'mean', 'median', np.std, 'min', 'max'],
        'Tot_30day_Fills': ['sum', 'mean', 'median', np.std, 'min', 'max'],
        'Tot_Day_Suply': ['sum', 'mean', 'median', np.std, 'min', 'max'],
        'Tot_Drug_Cst': ['sum', 'mean', 'median', np.std, 'min', 'max']
    })
    print(f"Gruplanmış veri boyutu: {partd_main.shape}")

Gruplanmış veri boyutu: (798481, 30)


In [12]:
if partd_main is not None:
    # MultiIndex sütun adlarını düzleştir
    partd_main.columns = ['_'.join(col) for col in partd_main.columns.values]
    
    # Index'i sıfırla
    partd_main = partd_main.reset_index(level=['Prscrbr_Type', 'npi', 'DATA_YEAR'])
    
    print("Sütun adları düzleştirildi.")
    print(f"Yeni sütunlar: {partd_main.columns.tolist()[:10]}...")

Sütun adları düzleştirildi.
Yeni sütunlar: ['npi', 'Prscrbr_Type', 'DATA_YEAR', 'Tot_Benes_sum', 'Tot_Benes_mean', 'Tot_Benes_median', 'Tot_Benes_std', 'Tot_Benes_min', 'Tot_Benes_max', 'Tot_Clms_sum']...


## 3. Medicare Part D ve LEIE Birleştirme

In [13]:
if partd_main is not None and leie is not None:
    # NPI üzerinden outer join ile birleştir
    partd = pd.merge(filtered_leie, partd_main, on='npi', how='outer')
    print(f"Birleştirilmiş veri boyutu: {partd.shape}")

Birleştirilmiş veri boyutu: (870222, 50)


In [14]:
if 'partd' in dir():
    # Gerekli sütunları seç ve yeniden sırala
    partd = partd[[
        'npi', 'Prscrbr_Type', 'DATA_YEAR',       
        'Tot_Benes_sum', 'Tot_Benes_mean', 'Tot_Benes_median',
        'Tot_Benes_std', 'Tot_Benes_min', 'Tot_Benes_max',
        'Tot_Clms_sum', 'Tot_Clms_mean', 'Tot_Clms_median', 
        'Tot_Clms_std', 'Tot_Clms_min', 'Tot_Clms_max',
        'Tot_30day_Fills_sum', 'Tot_30day_Fills_mean',
        'Tot_30day_Fills_median', 'Tot_30day_Fills_std',
        'Tot_30day_Fills_min', 'Tot_30day_Fills_max',
        'Tot_Day_Suply_sum', 'Tot_Day_Suply_mean',
        'Tot_Day_Suply_median', 'Tot_Day_Suply_std',
        'Tot_Day_Suply_min', 'Tot_Day_Suply_max',
        'Tot_Drug_Cst_sum','Tot_Drug_Cst_mean', 
        'Tot_Drug_Cst_median', 'Tot_Drug_Cst_std',
        'Tot_Drug_Cst_min', 'Tot_Drug_Cst_max', 
        'EXCLTYPE', 'EXCLDATE', 'WVRSTATE'
    ]]
    print(f"Sütun sayısı: {len(partd.columns)}")

Sütun sayısı: 36


### NPI veya Provider olmayan satırları kaldır

In [15]:
if 'partd' in dir():
    # Geçersiz satırları kaldır
    partd = partd[(partd.npi != 0) & (partd.Prscrbr_Type.notna())]
    print(f"Filtrelenmiş veri boyutu: {partd.shape}")

Filtrelenmiş veri boyutu: (798496, 36)


### Hedef Değişken (TARGET) Oluştur

In [16]:
if 'partd' in dir():
    # Varsayılan olarak NOT_FRAUD
    partd['TARGET'] = 'NOT_FRAUD'
    
    # EXCLDATE'den yıl çıkar
    partd['START_EXCLDATE'] = partd['EXCLDATE'].dt.year
    print("TARGET ve START_EXCLDATE sütunları oluşturuldu.")

TARGET ve START_EXCLDATE sütunları oluşturuldu.


In [17]:
def make_labels(start_exc, data_yr):
    """
    Hariç tutma yılı ve veri yılına göre fraud etiketi oluşturur.
    
    Args:
        start_exc: Hariç tutma başlangıç yılı
        data_yr: Veri yılı
    
    Returns:
        'FRAUD' veya 'NOT_FRAUD'
    """
    if pd.isna(start_exc) or pd.isna(data_yr):
        return 'NOT_FRAUD'
    if start_exc > data_yr:
        return 'FRAUD'  # Hariç tutma veri yılından sonra = o yıl fraud yapıyordu
    else:
        return 'NOT_FRAUD'

In [18]:
if 'partd' in dir():
    # TARGET sütununu oluştur
    partd['TARGET'] = partd[['START_EXCLDATE', 'DATA_YEAR']].apply(
        lambda x: make_labels(x['START_EXCLDATE'], x['DATA_YEAR']), axis=1
    )
    
    print("\nTARGET dağılımı:")
    print(partd.TARGET.value_counts())


TARGET dağılımı:
TARGET
NOT_FRAUD    797540
FRAUD           956
Name: count, dtype: int64


## 4. Veri Setini Küçültme (Downsampling)

In [20]:
if 'partd' in dir():
    # Hedef boyut: 1 milyon satır
    desired_total = 1_000_000

    # Orijinal oranı koru
    fraud_ratio = partd['TARGET'].value_counts(normalize=True)['FRAUD']
    not_fraud_ratio = partd['TARGET'].value_counts(normalize=True)['NOT_FRAUD']

    num_fraud = int(desired_total * fraud_ratio)
    num_not_fraud = desired_total - num_fraud

    print(f"{num_fraud} FRAUD ve {num_not_fraud} NOT_FRAUD örnekleniyor.")

    # Sınıfları ayır
    df_fraud = partd[partd['TARGET'] == 'FRAUD']
    df_not_fraud = partd[partd['TARGET'] == 'NOT_FRAUD']

    # Örnekle
    df_fraud_sampled = df_fraud.sample(n=min(num_fraud, len(df_fraud)), random_state=42)
    df_not_fraud_sampled = df_not_fraud.sample(n=min(num_not_fraud, len(df_not_fraud)), random_state=42)

    # Birleştir ve karıştır
    df_sampled = pd.concat([df_fraud_sampled, df_not_fraud_sampled])
    df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nYeni TARGET dağılımı:")
    print(df_sampled['TARGET'].value_counts())
    print(f"\nYeni veri seti boyutu: {len(df_sampled)}")

1197 FRAUD ve 998803 NOT_FRAUD örnekleniyor.

Yeni TARGET dağılımı:
TARGET
NOT_FRAUD    797540
FRAUD           956
Name: count, dtype: int64

Yeni veri seti boyutu: 798496


## 5. Kategorik Sütunlar için One-Hot Encoding

In [21]:
if 'df_sampled' in dir():
    # Kategorik sütunlar
    partd_category_columns = ['Prscrbr_Type']
    
    # One-hot encoding uygula
    df_sampled = pd.get_dummies(df_sampled, columns=partd_category_columns, drop_first=True)
    print(f"One-hot encoding sonrası sütun sayısı: {len(df_sampled.columns)}")

One-hot encoding sonrası sütun sayısı: 221


In [22]:
if 'df_sampled' in dir():
    # Gereksiz sütunları kaldır
    columns_to_drop = ['EXCLTYPE', 'EXCLDATE', 'WVRSTATE', 'START_EXCLDATE', 'npi', 'DATA_YEAR']
    df_sampled.drop(columns_to_drop, axis=1, inplace=True, errors='ignore')
    
    # Eksik değerleri 0 ile doldur
    df_sampled.fillna(0, inplace=True)
    
    print(f"Son veri seti boyutu: {df_sampled.shape}")
    df_sampled.head()

Son veri seti boyutu: (798496, 215)


## 6. İşlenmiş Veriyi Kaydet

In [23]:
if 'df_sampled' in dir():
    # CSV olarak kaydet
    output_path = "../data/Combined_LEIE_Medicare_2017_2019_DOWNSIZED_1mil.csv"
    df_sampled.to_csv(output_path, index=False)
    print(f"✓ Veri seti kaydedildi: {output_path}")
    print(f"  → Boyut: {df_sampled.shape}")
    print(f"  → Sütunlar: {len(df_sampled.columns)}")

✓ Veri seti kaydedildi: ../data/Combined_LEIE_Medicare_2017_2019_DOWNSIZED_1mil.csv
  → Boyut: (798496, 215)
  → Sütunlar: 215


## 7. Mevcut İşlenmiş Veriyi Kontrol Et

Eğer yukarıdaki adımları atlarsanız, hazır veri setini kontrol edin:

In [ ]:
import os

# İşlenmiş veri seti konumu
processed_file = "../data/Combined_LEIE_Medicare_2017_2019_DOWNSIZED_1mil.csv"

if os.path.exists(processed_file):
    file_size = os.path.getsize(processed_file) / (1024**3)  # GB
    print(f"✓ İşlenmiş veri seti mevcut: {processed_file}")
    print(f"  → Dosya boyutu: {file_size:.2f} GB")
    
    # İlk satırları kontrol et
    df_check = pd.read_csv(processed_file, nrows=5)
    print(f"\nSütun sayısı: {len(df_check.columns)}")
    print(f"\nÖrnek veriler:")
    print(df_check.head())
else:
    print(f"⚠ İşlenmiş veri bulunamadı: {processed_file}")